# Automatically Select the Best Imputation Strategy Using GridSearchCV

---

## 🧾 Notebook Summary:

This notebook demonstrates how to build a robust preprocessing and modeling pipeline that **automatically selects the best imputation strategy** using `GridSearchCV`. Instead of manually choosing between strategies like `"mean"`, `"median"`, or `"most_frequent"`, we let the model find the best combination by testing all options through **cross-validation**.

By combining:

* `SimpleImputer` for handling missing values
* `ColumnTransformer` for separate treatment of numerical and categorical columns
* `Pipeline` for organizing preprocessing + modeling
* `GridSearchCV` for hyperparameter tuning

…you get a powerful and flexible setup that’s easy to scale and maintain.

---

### 🔍 What’s the Goal?

When your dataset has missing values, you usually fill them using one of the following:

* The **mean** (average)
* The **median** (middle value)
* The **most frequent** category
* A **constant** like `"missing"`

Instead of guessing which one works best, we use `GridSearchCV` to:

1. Try all reasonable imputation strategies
2. Train models with each strategy
3. Automatically select the best-performing (aka **Accuracy score**) combination

---

📌 **Useful Techniques demoed**

1. How to create full ML pipeline for pre-processing and training a model (a logistic regression model in this case)
2. The pre-processing pipeline is for numerical column and categorical columns
3. Visualizing the pipeline
4. Automatically find the best combination of imputation strategies and model hyperparameters using **GridSearchCV** to maximize prediction accuracy.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [5]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
df.isnull().mean() * 100

,0
Survived,0.000000
Pclass,0.000000
Sex,0.000000
Age,19.865320
SibSp,0.000000
Parch,0.000000
Fare,0.000000
Embarked,0.224467


Missing values in Age and Embarked

In [7]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [9]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
30,1,male,40.0,0,0,27.7208,C
10,3,female,4.0,1,1,16.7000,S
873,3,male,47.0,0,0,9.0000,S
182,3,male,9.0,4,2,31.3875,S
876,3,male,20.0,0,0,9.8458,S


## **Apply Pipelines**

---

## 🎯 Objective

We **prepare raw data for machine learning models** by building a **two-part preprocessing pipeline**—one for **numerical features** and one for **categorical features**.

---

### ✅ Why This Is Needed

Real-world datasets often contain:

* **Missing values** (e.g., Age is unknown)
* **Unscaled numerical data** (e.g., Fare ranges from 0 to hundreds)
* **Text/categorical data** (e.g., "Sex" is "male" or "female")

These issues must be resolved before fitting models like **Logistic Regression**, which expects clean, numeric input with similar scales.

---

### 🛠️ What the Code Does

1. **Numerical pipeline**:

   * Fills missing values using the **median** (robust to outliers)
   * Standardizes the values (mean = 0, std = 1)

2. **Categorical pipeline**:

   * Fills missing values with the **most frequent value**
   * Converts text categories into **binary numeric vectors** using **one-hot encoding**

---

### 📦 Result

The output of these pipelines will be a **fully numeric, clean, and model-ready dataset**.

You’ll later plug these pipelines into a `ColumnTransformer`, and then into a full `Pipeline` with a classifier (e.g., `LogisticRegression`), allowing you to:

* Automate preprocessing
* Tune everything with cross-validation (`GridSearchCV`)
* Deploy consistently clean input during prediction

---

In [10]:
# List of numeric columns we want to process
numerical_features = ['Age', 'Fare']

# Create a pipeline for numerical features
numerical_transformer = Pipeline(steps=[
    # Step 1: Fill missing values with the median of each column
    ('imputer', SimpleImputer(strategy='median')),

    # Step 2: Standardize values (mean=0, std=1) to help some models perform better
    ('scaler', StandardScaler())
])

# List of categorical (non-numeric) columns we want to process
categorical_features = ['Embarked', 'Sex']

# Create a pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    # Step 1: Fill missing values with the most frequent (mode) value in each column
    ('imputer', SimpleImputer(strategy='most_frequent')),

    # Step 2: Convert categorical values to binary columns (one-hot encoding)
    # 'ignore' prevents errors if unseen categories appear during prediction
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

In [11]:
# Combine both numerical and categorical preprocessing steps into one preprocessor
# This allows you to apply different transformations to different column types

preprocessor = ColumnTransformer(
    transformers=[
        # Apply the numerical_transformer to the numerical_features
        ('num', numerical_transformer, numerical_features),

        # Apply the categorical_transformer to the categorical_features
        ('cat', categorical_transformer, categorical_features)
    ]
)

---

### 🧠 What This Does:

* The `ColumnTransformer` makes it easy to apply **different preprocessing pipelines** to **different subsets of columns** in your dataset.
* You can then plug this `preprocessor` into a full ML pipeline (e.g., with a model like `LogisticRegression` or `RandomForestClassifier`) to streamline the entire workflow.

---


In [12]:
# Create a full ML pipeline that includes:
# 1. Preprocessing (handling missing values, scaling, encoding)
# 2. Training a logistic regression model

clf = Pipeline(steps=[
    # Step 1: Apply the preprocessor defined earlier (ColumnTransformer)
    # This handles both numerical and categorical data
    ('preprocessor', preprocessor),

    # Step 2: Train a logistic regression model on the transformed data
    ('classifier', LogisticRegression())
])

In [13]:
from sklearn import set_config

# This config tells scikit-learn to display pipeline components as a visual diagram
# (instead of plain text) when viewed in a Jupyter notebook or IPython
set_config(display='diagram')

# Display the full pipeline (clf) in a neat, tree-like structure
clf

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Embarked', 'Sex'])])),
                ('classifier', LogisticRegression())])

### **Run GridSearchCV to find the best combination of hyperparameters**

In [14]:
# -----------------------------------------------
# Define the grid of hyperparameters to search over
# -----------------------------------------------

param_grid = {
    # Try both 'mean' and 'median' strategies for imputing numerical columns
    'preprocessor__num__imputer__strategy': ['mean', 'median'],

    # Try 'most_frequent' and 'constant' (like 'missing') for categorical columns
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],

    # Try different regularization strengths for LogisticRegression (C = 1/λ)
    # Smaller C → stronger regularization, Larger C → weaker regularization
    'classifier__C': [0.1, 1.0, 10, 100]
}

# -----------------------------------------------
# Run GridSearchCV to find the best combination
# -----------------------------------------------

from sklearn.model_selection import GridSearchCV

# Use cross-validation (cv=10 means 10-fold cross-validation)
grid_search = GridSearchCV(clf, param_grid, cv=10)
# ➤ This will:
#    - Try all combinations from param_grid
#    - Train and evaluate each one using 10-fold CV
#    - Pick the best combination based on default scoring (accuracy)

---

### 🔍 Using `GridSearchCV` to Find the Optimal Hyperparameters

---

### 🎯 Objective

The goal of this section is to **automatically find the best combination of preprocessing strategies and model settings** (called *hyperparameters*) using **cross-validation**.

---

### 🧠 What is `GridSearchCV`?

`GridSearchCV` is a tool from `scikit-learn` that:

1. Takes in a **range of possible values** for multiple hyperparameters
2. Trains and evaluates the model on each combination using **cross-validation**
3. Picks the combination that performs **best on average** across the validation folds

---

### ⚙️ What It Does in This Notebook

In your notebook, `GridSearchCV` is applied to the **full preprocessing + modeling pipeline** (`Pipeline`). It searches over:

| Component                      | Hyperparameter                | Values Tried                    |
| ------------------------------ | ----------------------------- | ------------------------------- |
| Numerical Imputer              | `strategy`                    | `'mean'`, `'median'`            |
| Categorical Imputer            | `strategy`                    | `'most_frequent'`, `'constant'` |
| Logistic Regression Classifier | `C` (regularization strength) | `0.1`, `1.0`, `10`, `100`       |

All possible combinations of these values are tested, and their performance is measured using **10-fold cross-validation**.

---

### ✅ Why It's Useful

* Saves you from guessing which imputation strategy or model setting is best
* Ensures that the final model is trained on the configuration that performs **best on unseen data**
* Makes your pipeline more **robust and generalizable**

---

### 📈 Example Outcome

```text
Best parameters:
{'classifier__C': 1.0,
 'preprocessor__cat__imputer__strategy': 'most_frequent',
 'preprocessor__num__imputer__strategy': 'mean'}

Internal CV score: 0.788
```

This tells us:

* The model performed best using:

  * Mean imputation for numerical columns
  * Most frequent value imputation for categorical columns
  * A regularization strength of `C = 1.0`
* The model correctly predicted the target about **78.8% of the time** during internal cross-validation.

---

Let me know if you want this inserted into your notebook as a markdown cell or turned into a visual chart showing the grid search results!


---


In [15]:
grid_search.fit(X_train, y_train)

print(f"Best params:")
print(grid_search.best_params_)

Best params:
{'classifier__C': 1.0, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}


---

### ✅ Best Hyperparameter Combination Found by `GridSearchCV`

```python
Best params:
{
  'classifier__C': 1.0,
  'preprocessor__cat__imputer__strategy': 'most_frequent',
  'preprocessor__num__imputer__strategy': 'mean'
}
```

---

### 🔍 What Each Parameter Means:

| Parameter                                                | Meaning                                                       | Why It Matters                                                                            |
| -------------------------------------------------------- | ------------------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| `classifier__C = 1.0`                                    | Regularization strength in `LogisticRegression`               | A moderate value of `C` balances underfitting and overfitting.                            |
| `preprocessor__cat__imputer__strategy = 'most_frequent'` | Fill missing categorical values with the most frequent (mode) | This keeps imputation simple and usually works well when categories are repeated.         |
| `preprocessor__num__imputer__strategy = 'mean'`          | Fill missing numerical values with the mean                   | The mean worked better than median here for your dataset (perhaps due to fewer outliers). |

---

### 🧠 Summary:

* Your model achieved **best cross-validation performance** using this configuration.
* This is the combination you should use for final training and testing.
* Now you can use:

  ```python
  grid_search.best_estimator_  
  ```

  to make predictions or evaluate on your test set.

---

In [16]:
print(f"Internal CV score: {grid_search.best_score_:.3f}")

Internal CV score: 0.788


---

### 🧠 What This Score Means:

* **`0.788`** is the **average accuracy** across all 10 folds of cross-validation for the **best combination** of parameters found by `GridSearchCV`.
* In simpler terms, when the training data was split into 10 parts and the model was trained/tested 10 times:

  * It predicted correctly about **78.8% of the time** on unseen (validation) data.

---

### 🧪 Why It’s Important:

* This is an estimate of how well your model **generalizes** to unseen data.
* It helps you **select the best configuration** before touching your test set (which should be used only once at the very end).

---

In [17]:
import pandas as pd

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)

cv_results[['param_classifier__C','param_preprocessor__cat__imputer__strategy','param_preprocessor__num__imputer__strategy','mean_test_score']]

,param_classifier__C,param_preprocessor__cat__imputer__strategy,param_preprocessor__num__imputer__strategy,mean_test_score
7,1.0,constant,median,0.787852
6,1.0,constant,mean,0.787852
5,1.0,most_frequent,median,0.787852
4,1.0,most_frequent,mean,0.787852
11,10.0,constant,median,0.787852
10,10.0,constant,mean,0.787852
9,10.0,most_frequent,median,0.787852
8,10.0,most_frequent,mean,0.787852
12,100.0,most_frequent,mean,0.787852
13,100.0,most_frequent,median,0.787852


---

## 📊 Interpreting GridSearchCV Results

### ✅ What This Table Shows:

This DataFrame shows all combinations of the following parameters that were tried:

* `classifier__C`: Regularization strength
* `preprocessor__cat__imputer__strategy`: Strategy for imputing categorical values
* `preprocessor__num__imputer__strategy`: Strategy for imputing numerical values
* `mean_test_score`: Average cross-validation accuracy for that combination

### 🏆 Best Results

Notice that **many combinations** gave the **same top score**:

```text
mean_test_score ≈ 0.787852
```

This means:

* Several combinations performed **equally well** on cross-validation.
* The best `C` value (regularization) appears to be between **1.0 and 100.0**.
* Both `'mean'` and `'median'` strategies worked similarly for numerical imputation.
* Both `'most_frequent'` and `'constant'` were equally good for categorical imputation.

---

### 🧠 What You Can Conclude:

* The **model is robust** to small changes in imputation strategy.
* No need to over-optimize – simple imputation choices (like `'mean'` and `'most_frequent'`) work well.
* Logistic Regression performs best when `C = 1.0` or higher (weaker regularization).

---


## 📌 Key Takeways of this notebook

## ✅ When to Use This Technique

| Use This Technique When...                                            | Avoid It When...                                                     |
| --------------------------------------------------------------------- | -------------------------------------------------------------------- |
| You have **missing values** in both numeric & categorical columns     | Your dataset has **no missing values** at all                        |
| You are unsure which imputation strategy works best                   | You already have a domain-specific rule (e.g., Age=0 is valid)       |
| You are building a **scikit-learn pipeline** and want full automation | You are using models that handle missingness natively (like XGBoost) |

---

## ⚙️ Best Suited For These Models

This technique works best with models that:

* **Do not handle missing values internally**
* Require clean, numeric inputs

Examples:

* `LogisticRegression`
* `LinearRegression`
* `SVM`, `KNN`
* `MLPClassifier`

Not as critical for:

* **Tree-based models** (like Random Forest or XGBoost), which can internally handle missing values by learning split directions for `NaN`s.